# ADVNT Training Template

This notebook assumes you will load data into `x_train`, `y_train`, and `x_test`. After that, run adversarial validation, inspect drift artifacts, and fit either a classifier or regressor.

In [ ]:
# Core scientific stack
import numpy as np
import pandas as pd

# Display helpers
from IPython.display import display

# sklearn utilities
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ADVNT public API
from advnt import (
    AdversarialValidator,
    extract_model_importances_from_train_test,
    compute_density_ratio_weights,
    compute_density_ratio_weights_from_train_test,
    neutralize_features,
    select_safe_pseudo_labels,
    select_safe_pseudo_labels_from_train_test,
    run_adversarial_validation_workflow,
    run_shift_preparation_workflow,
    AdversarialValidationMLPClassifier,
    AdversarialValidationMLPRegressor,
    GradientReversalLayer,
)

# ADVNT module-level helpers, useful while developing locally
from advnt.validation import AdversarialValidator
from advnt.sample_weights import compute_density_ratio_weights
from advnt.ssl import select_safe_pseudo_labels
from advnt.workflows import run_adversarial_validation_workflow, run_shift_preparation_workflow

RANDOM_STATE = 42
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

## Load Your Data

Load your data before continuing. The expected variables are:

- `x_train`: training features
- `y_train`: training target
- `x_test`: test or inference features

In [ ]:
# Example only. Replace this cell with your own data-loading code.
# x_train = pd.read_csv("train_features.csv")
# y_train = pd.read_csv("train_target.csv").squeeze("columns")
# x_test = pd.read_csv("test_features.csv")

In [ ]:
required_names = ["x_train", "y_train", "x_test"]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise NameError(f"Load these variables before continuing: {missing}")

print(f"x_train shape: {getattr(x_train, 'shape', None)}")
print(f"y_train shape: {getattr(y_train, 'shape', None)}")
print(f"x_test shape:  {getattr(x_test, 'shape', None)}")

if hasattr(x_train, "head"):
    display(x_train.head())

## Adversarial Validation

In [ ]:
# Pass an explicit sklearn model so this works without optional LightGBM installed.
adversarial_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

av = AdversarialValidator(
    model=adversarial_model,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    metric="roc_auc",
    random_state=RANDOM_STATE,
    compute_sample_weights=True,
    weight_clip=(0.01, 100.0),
    normalize_weights=True,
)

av.fit(x_train, x_test)

print(f"Adversarial validation AUC: {av.score_:.4f}")
print("Fold scores:", np.round(av.fold_scores_, 4))
print(f"Sample weights mean: {np.mean(av.sample_weights_):.4f}")
print(f"Sample weights min/max: {np.min(av.sample_weights_):.4f} / {np.max(av.sample_weights_):.4f}")

In [ ]:
if av.feature_importances_ is not None:
    display(av.feature_importances_.head(20))
else:
    print("No model-based feature importances are available for this adversarial model.")

In [ ]:
# Rows with very low or very high test-domain probability are confident domain pseudo-label candidates.
safe_pseudo_label_info = select_safe_pseudo_labels(av.test_proba_, threshold=0.10)
safe_test_mask = safe_pseudo_label_info["mask"]
print(f"Safe pseudo-label candidates: {safe_test_mask.sum()} / {len(safe_test_mask)}")
display(pd.DataFrame({
    "index": safe_pseudo_label_info["indices"],
    "pseudo_label": safe_pseudo_label_info["pseudo_labels"],
    "confidence": safe_pseudo_label_info["confidence"],
}).head())

## Fit Final Classifier Or Regressor

In [ ]:
# Choose one: "classifier" or "regressor"
TASK = "classifier"

# Use ADVNT covariate-shift weights when fitting the supervised model.
USE_ADV_SAMPLE_WEIGHTS = True

numeric_features = make_column_selector(dtype_include=np.number)
categorical_features = make_column_selector(dtype_exclude=np.number)

numeric_preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocess, numeric_features),
        ("cat", categorical_preprocess, categorical_features),
    ],
    remainder="drop",
)

if TASK == "classifier":
    estimator = RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
elif TASK == "regressor":
    estimator = RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
else:
    raise ValueError('TASK must be either "classifier" or "regressor".')

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", estimator),
    ]
)

fit_params = {}
if USE_ADV_SAMPLE_WEIGHTS and av.sample_weights_ is not None:
    fit_params["model__sample_weight"] = av.sample_weights_

model.fit(x_train, y_train, **fit_params)
print(f"Fitted {TASK}: {model.named_steps['model'].__class__.__name__}")

## Optional Holdout Check

In [ ]:
x_fit, x_valid, y_fit, y_valid, w_fit, w_valid = train_test_split(
    x_train,
    y_train,
    av.sample_weights_ if av.sample_weights_ is not None else np.ones(len(x_train)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train if TASK == "classifier" else None,
)

holdout_model = clone(model)
holdout_fit_params = {}
if USE_ADV_SAMPLE_WEIGHTS:
    holdout_fit_params["model__sample_weight"] = w_fit

holdout_model.fit(x_fit, y_fit, **holdout_fit_params)

if TASK == "classifier":
    valid_pred = holdout_model.predict(x_valid)
    print(classification_report(y_valid, valid_pred))
    if hasattr(holdout_model.named_steps["model"], "predict_proba") and len(np.unique(y_valid)) == 2:
        valid_proba = holdout_model.predict_proba(x_valid)[:, 1]
        print(f"ROC AUC: {roc_auc_score(y_valid, valid_proba):.4f}")
        print(f"Average precision: {average_precision_score(y_valid, valid_proba):.4f}")
else:
    valid_pred = holdout_model.predict(x_valid)
    rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
    print(f"MAE:  {mean_absolute_error(y_valid, valid_pred):.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2:   {r2_score(y_valid, valid_pred):.4f}")

## Predict On `x_test`

In [ ]:
if TASK == "classifier" and hasattr(model.named_steps["model"], "predict_proba"):
    test_predictions = model.predict(x_test)
    test_probabilities = model.predict_proba(x_test)
    display(pd.DataFrame(test_probabilities).head())
else:
    test_predictions = model.predict(x_test)

submission = pd.DataFrame({"prediction": test_predictions})
display(submission.head())